# Option A: Out-of-Sample Projection Drift (AE vs UMAP)

**Experiment Design:**
1. Randomly hold out 10% of OAs (~19,000 areas)
2. Train AE and UMAP on remaining 90%
3. Project held-out 10% using:
   - AE: encoder forward pass
   - UMAP: transform() method
4. Retrain both methods on all 100% of OAs
5. Compare how much embeddings shifted (after Procrustes alignment)

**Expected Result:**
- **AE**: Projected embeddings ≈ full-retrain embeddings (stable)
- **UMAP**: Large drift because global manifold structure shifts with new points

**Real-world relevance:** Simulates adding new census data (e.g., Scotland releases 50,000 new OAs)

## 1. Setup and Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import yaml
from torchgeodemo import autoencoder_train_latent
from sklearn.model_selection import train_test_split
from sklearn.metrics import pairwise_distances
from scipy.spatial import procrustes
from scipy.stats import pearsonr, spearmanr
import umap
import os
import pickle
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# Set random seeds for reproducibility
random_seed = 20210321
np.random.seed(random_seed)
torch.manual_seed(random_seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(random_seed)

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

## 2. Configuration

In [ ]:
# Helper function for layer size generation (from notebook 2)
def gen_layer_sizes(input_size, latent_size, num_layers, scaling_type="lin"):
    """Generate layer sizes for encoder/decoder"""
    if scaling_type == "mul":
        # Geometric scaling
        scale_factor = (latent_size / input_size) ** (1 / (num_layers - 1))
        layer_sizes = [int(input_size * scale_factor ** i) for i in range(num_layers)]
    elif scaling_type == "lin":
        # Linear scaling
        step = (latent_size - input_size) / (num_layers - 1)
        layer_sizes = [int(input_size + step * i) for i in range(num_layers)]
    else:
        raise ValueError("Invalid scaling type. Use 'mul' or 'lin'.")
    
    # Exclude input layer for torchgeodemo
    layer_sizes = layer_sizes[1:]
    return layer_sizes

# Paths
data_path = "../data/census_data/engcensus_cleaned_scaled.parquet"
output_dir = "./plots/projection_drift/"
os.makedirs(output_dir, exist_ok=True)
os.makedirs(f"{output_dir}/data/", exist_ok=True)
os.makedirs(f"{output_dir}/models/", exist_ok=True)
os.makedirs(f"{output_dir}/yamls/", exist_ok=True)

# Experiment parameters
latent_dim = 100  # Use 100D linear AE
holdout_fraction = 0.1  # Hold out 10%
n_epochs = 250  # Match notebook 2
batch_size = 0.01  # Match notebook 2 (1% of data)
scaling_type = "lin"  # Linear layer scaling

# UMAP parameters
umap_n_neighbors = 15
umap_min_dist = 0.1
umap_metric = 'euclidean'

print(f"Configuration:")
print(f"  Latent dimension: {latent_dim}")
print(f"  Holdout fraction: {holdout_fraction} ({int(holdout_fraction*100)}%)")
print(f"  Epochs: {n_epochs}")
print(f"  Batch size: {batch_size} ({int(batch_size*100)}% of data)")
print(f"  Layer scaling: {scaling_type}")
print(f"  UMAP n_neighbors: {umap_n_neighbors}")
print(f"  UMAP min_dist: {umap_min_dist}")
print(f"  Output directory: {output_dir}")

## 3. Load Data

In [ ]:
# Load census data
df = pd.read_parquet(data_path)
df = df.set_index('OA')
print(f"Census data shape: {df.shape}")
print(f"Total OAs: {len(df)}")
print(f"Variables: {len(df.columns)}")

# Convert to numpy for easier manipulation
X_full = df.values
oa_ids = df.index.values

print(f"\nData summary:")
print(f"  Shape: {X_full.shape}")
print(f"  Data range: [{X_full.min():.4f}, {X_full.max():.4f}]")

## 4. Helper Functions for AE Training via YAML

In [ ]:
def train_ae_via_yaml(data_path, X_subset, subset_name, latent_dim, working_dir, 
                      n_epochs=250, batch_size=0.01, scaling_type="lin"):
    """
    Train autoencoder using torchgeodemo with YAML configuration.
    
    Parameters:
    - data_path: path to full dataset parquet
    - X_subset: subset of data to train on (pandas DataFrame with 'OA' index)
    - subset_name: name for this training run (e.g., '90pct', '100pct')
    - latent_dim: bottleneck dimension
    - working_dir: directory for outputs
    - n_epochs: training epochs
    - batch_size: batch size fraction
    - scaling_type: 'lin' or 'mul' for layer scaling
    
    Returns: path to trained model
    """
    # Generate layer sizes (4 layers total including input and latent)
    input_dim = X_subset.shape[1]
    encoder_sizes = gen_layer_sizes(input_dim, latent_dim, num_layers=4, scaling_type=scaling_type)
    
    # Create YAML configuration
    yaml_config = {
        "data": {
            "source": data_path,
            "nickname": f"drift_exp_{subset_name}",
            "id_col": "OA"
        },
        "working_dir": working_dir,
        "autoencoder": {
            "nickname": f"ae_{latent_dim}d_{subset_name}",
            "version": "1",
            "save_latent": "csv",
            "max_epochs": n_epochs,
            "batch_size": batch_size,
            "use_covariance_loss": False,
            "encoder": {
                "sizes": encoder_sizes,
                "activation": "LeakyReLU"
            },
            "decoder": {
                "sizes": encoder_sizes[::-1],  # Reverse for decoder
                "activation": "LeakyReLU"
            }
        }
    }
    
    # Save YAML config
    yaml_dir = f"{working_dir}/yamls"
    os.makedirs(yaml_dir, exist_ok=True)
    config_path = f"{yaml_dir}/config_{subset_name}_{latent_dim}d.yaml"
    
    with open(config_path, 'w') as f:
        yaml.dump(yaml_config, f, default_flow_style=False)
    
    print(f"\n{'='*80}")
    print(f"Training AE on {subset_name} subset ({len(X_subset)} OAs)")
    print(f"  Config: {config_path}")
    print(f"  Encoder sizes: {[input_dim] + encoder_sizes}")
    print(f"  Decoder sizes: {encoder_sizes[::-1] + [input_dim]}")
    print(f"{'='*80}\n")
    
    # Train model using torchgeodemo
    # Note: We need to write subset data to temp file for torchgeodemo to load
    temp_data_path = f"{working_dir}/temp_data_{subset_name}.parquet"
    X_subset_with_index = X_subset.reset_index()  # Ensure OA column exists
    X_subset_with_index.to_parquet(temp_data_path)
    
    # Update config to point to temp data
    yaml_config["data"]["source"] = temp_data_path
    with open(config_path, 'w') as f:
        yaml.dump(yaml_config, f, default_flow_style=False)
    
    # Train
    autoencoder_train_latent.main(config_path, create_latent=True, save_reco_error=True, verbose=True)
    
    # Return path to trained model
    model_name = f"drift_exp_{subset_name}_ae_{latent_dim}d_{subset_name}_v1"
    model_path = f"{working_dir}/{model_name}/{model_name}__model.pth"
    
    print(f"\n✓ Model trained and saved to: {model_path}\n")
    
    return model_path, config_path

print("YAML-based training function defined")

## 5. Procrustes Alignment Utility

In [ ]:
def procrustes_align(X_source, X_target):
    """
    Align X_source to X_target using Procrustes transformation.
    Handles rotation, reflection, and scaling.
    
    Returns: aligned X_source, disparity
    """
    mtx1, mtx2, disparity = procrustes(X_target, X_source)
    return mtx1, disparity

def compute_drift_metrics(embedding_projected, embedding_full, oa_ids_subset):
    """
    Compute drift metrics between projected and full-retrain embeddings.
    
    Returns dict with:
    - mean_euclidean_distance
    - per_dimension_correlation
    - mean_correlation
    """
    # Align projected to full using Procrustes
    aligned_projected, disparity = procrustes_align(embedding_projected, embedding_full)
    
    # Compute Euclidean distances
    distances = np.linalg.norm(aligned_projected - embedding_full, axis=1)
    mean_distance = distances.mean()
    
    # Compute per-dimension correlations
    n_dims = embedding_projected.shape[1]
    dim_correlations = []
    for dim in range(n_dims):
        corr, _ = pearsonr(aligned_projected[:, dim], embedding_full[:, dim])
        dim_correlations.append(corr)
    
    return {
        'mean_euclidean_distance': mean_distance,
        'dim_correlations': np.array(dim_correlations),
        'mean_correlation': np.mean(dim_correlations),
        'procrustes_disparity': disparity,
        'distances_per_oa': distances,
        'aligned_projected': aligned_projected
    }

print("Procrustes alignment utilities defined")

## 5. Helper Functions for Model Loading and Encoding

In [ ]:
def load_ae_and_encode(model_path, X_data):
    """
    Load trained AE model and encode data.
    
    Parameters:
    - model_path: path to saved .pth model
    - X_data: numpy array or DataFrame to encode
    
    Returns: encoded embeddings (numpy array)
    """
    from torchgeodemo.models import AutoEncoder
    
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    
    # Load model
    model = torch.load(model_path, map_location=device, weights_only=False)
    model.eval()
    
    # Convert to numpy if needed
    if isinstance(X_data, pd.DataFrame):
        X_np = X_data.values
    else:
        X_np = X_data
    
    # Encode
    with torch.no_grad():
        X_tensor = torch.FloatTensor(X_np).to(device)
        embeddings = model.encode(X_tensor).cpu().numpy()
    
    return embeddings

print("Model loading and encoding functions defined")

## 6. Split Data: 90% Train, 10% Holdout

In [ ]:
# Load full dataset as DataFrame
df = pd.read_parquet(data_path)
print(f"Full dataset shape: {df.shape}")

# Ensure OA is in the dataframe (not just index) for torchgeodemo
if 'OA' not in df.columns:
    df = df.reset_index()

# Split indices
indices = np.arange(len(df))
train_indices, holdout_indices = train_test_split(
    indices, 
    test_size=holdout_fraction, 
    random_seed=random_seed
)

# Create subsets as DataFrames (keep OA column)
df_train_90 = df.iloc[train_indices]
df_holdout_10 = df.iloc[holdout_indices]

# Also create numpy versions for UMAP
X_train_90 = df_train_90.drop(columns=['OA']).values
X_holdout_10 = df_holdout_10.drop(columns=['OA']).values
X_full = df.drop(columns=['OA']).values

oa_ids_train = df_train_90['OA'].values
oa_ids_holdout = df_holdout_10['OA'].values
oa_ids = df['OA'].values

print(f"\nData split:")
print(f"  Training (90%): {len(df_train_90)} OAs")
print(f"  Holdout (10%): {len(df_holdout_10)} OAs")
print(f"  Total: {len(df)} OAs")
print(f"  Variables: {X_train_90.shape[1]}")

## 7. Experiment A1: Train AE on 90%, Project Holdout 10%

In [ ]:
print("=" * 80)
print("STEP 1: Train AE on 90% of data using torchgeodemo")
print("=" * 80)

# Train AE on 90% using YAML approach
model_path_90, config_path_90 = train_ae_via_yaml(
    data_path=data_path,
    X_subset=df_train_90,
    subset_name="90pct",
    latent_dim=latent_dim,
    working_dir=f"{output_dir}/models",
    n_epochs=n_epochs,
    batch_size=batch_size,
    scaling_type=scaling_type
)

# Load model and encode training data
ae_train_90_embeddings = load_ae_and_encode(model_path_90, X_train_90)
print(f"\nAE (90%) embeddings shape: {ae_train_90_embeddings.shape}")

# Project holdout 10% using trained encoder
ae_holdout_projected = load_ae_and_encode(model_path_90, X_holdout_10)
print(f"AE projected holdout embeddings shape: {ae_holdout_projected.shape}")

print(f"\n✓ AE trained on 90% and holdout 10% projected")

## 8. Experiment A2: Train UMAP on 90%, Project Holdout 10%

In [ ]:
print("=" * 80)
print("STEP 2: Train UMAP on 90% of data")
print("=" * 80)

# Train UMAP on 90%
umap_90 = umap.UMAP(
    n_components=latent_dim,
    n_neighbors=umap_n_neighbors,
    min_dist=umap_min_dist,
    metric=umap_metric,
    random_state=random_seed,
    verbose=True
)

print("Training UMAP (this may take several minutes)...")
umap_train_90_embeddings = umap_90.fit_transform(X_train_90)
print(f"UMAP (90%) embeddings shape: {umap_train_90_embeddings.shape}")

# Project holdout 10% using transform()
print("\nProjecting holdout 10% using UMAP.transform()...")
umap_holdout_projected = umap_90.transform(X_holdout_10)
print(f"UMAP projected holdout embeddings shape: {umap_holdout_projected.shape}")
print(f"\n✓ UMAP trained on 90% and holdout 10% projected")

## 9. Experiment B: Retrain Both Methods on 100% of Data

In [ ]:
print("=" * 80)
print("STEP 3: Retrain AE on 100% of data using torchgeodemo")
print("=" * 80)

# Train AE on 100% using YAML approach
model_path_100, config_path_100 = train_ae_via_yaml(
    data_path=data_path,
    X_subset=df,
    subset_name="100pct",
    latent_dim=latent_dim,
    working_dir=f"{output_dir}/models",
    n_epochs=n_epochs,
    batch_size=batch_size,
    scaling_type=scaling_type
)

# Load model and encode all data
ae_full_embeddings = load_ae_and_encode(model_path_100, X_full)

# Extract embeddings for holdout OAs from full training
ae_holdout_full = ae_full_embeddings[holdout_indices]

print(f"\nAE (100%) full embeddings shape: {ae_full_embeddings.shape}")
print(f"AE (100%) holdout subset shape: {ae_holdout_full.shape}")
print(f"\n✓ AE retrained on 100%")

In [ ]:
print("=" * 80)
print("STEP 4: Retrain UMAP on 100% of data")
print("=" * 80)

# Train UMAP on 100%
umap_100 = umap.UMAP(
    n_components=latent_dim,
    n_neighbors=umap_n_neighbors,
    min_dist=umap_min_dist,
    metric=umap_metric,
    random_state=random_seed,
    verbose=True
)

print("Training UMAP on full dataset (this may take several minutes)...")
umap_full_embeddings = umap_100.fit_transform(X_full)

# Extract embeddings for holdout OAs from full training
umap_holdout_full = umap_full_embeddings[holdout_indices]

print(f"\nUMAP (100%) full embeddings shape: {umap_full_embeddings.shape}")
print(f"UMAP (100%) holdout subset shape: {umap_holdout_full.shape}")
print(f"\n✓ UMAP retrained on 100%")

## 10. Compute Drift Metrics

In [ ]:
print("=" * 80)
print("STEP 5: Compute Drift Metrics")
print("=" * 80)

# Compute drift for AE
print("\nComputing drift metrics for AE...")
ae_drift = compute_drift_metrics(
    ae_holdout_projected, 
    ae_holdout_full,
    oa_ids_holdout
)

# Compute drift for UMAP
print("Computing drift metrics for UMAP...")
umap_drift = compute_drift_metrics(
    umap_holdout_projected,
    umap_holdout_full,
    oa_ids_holdout
)

print("\n" + "=" * 80)
print("DRIFT METRICS SUMMARY")
print("=" * 80)

print(f"\nAutoencoder (AE):")
print(f"  Mean Euclidean Distance: {ae_drift['mean_euclidean_distance']:.6f}")
print(f"  Mean Correlation: {ae_drift['mean_correlation']:.6f}")
print(f"  Procrustes Disparity: {ae_drift['procrustes_disparity']:.6f}")

print(f"\nUMAP:")
print(f"  Mean Euclidean Distance: {umap_drift['mean_euclidean_distance']:.6f}")
print(f"  Mean Correlation: {umap_drift['mean_correlation']:.6f}")
print(f"  Procrustes Disparity: {umap_drift['procrustes_disparity']:.6f}")

print(f"\nRatio (UMAP / AE):")
print(f"  Distance ratio: {umap_drift['mean_euclidean_distance'] / ae_drift['mean_euclidean_distance']:.2f}x")
print(f"  Correlation difference: {ae_drift['mean_correlation'] - umap_drift['mean_correlation']:.4f}")

# Save metrics
results = {
    'ae_drift': ae_drift,
    'umap_drift': umap_drift,
    'oa_ids_holdout': oa_ids_holdout
}

with open(f"{output_dir}/data/drift_metrics.pkl", 'wb') as f:
    pickle.dump(results, f)

print(f"\n✓ Metrics saved to {output_dir}/data/drift_metrics.pkl")

## 11. Visualization: Drift Comparison Table

In [ ]:
# Create comparison table
comparison_df = pd.DataFrame({
    'Metric': [
        'Mean Euclidean Distance',
        'Mean Correlation',
        'Procrustes Disparity',
        'Min Distance',
        'Max Distance',
        'Std Distance'
    ],
    'AE': [
        ae_drift['mean_euclidean_distance'],
        ae_drift['mean_correlation'],
        ae_drift['procrustes_disparity'],
        ae_drift['distances_per_oa'].min(),
        ae_drift['distances_per_oa'].max(),
        ae_drift['distances_per_oa'].std()
    ],
    'UMAP': [
        umap_drift['mean_euclidean_distance'],
        umap_drift['mean_correlation'],
        umap_drift['procrustes_disparity'],
        umap_drift['distances_per_oa'].min(),
        umap_drift['distances_per_oa'].max(),
        umap_drift['distances_per_oa'].std()
    ]
})

# Add ratio column
comparison_df['UMAP/AE Ratio'] = comparison_df['UMAP'] / comparison_df['AE']

print("\n" + "=" * 80)
print("DETAILED COMPARISON TABLE")
print("=" * 80)
print(comparison_df.to_string(index=False))

# Save table
comparison_df.to_csv(f"{output_dir}/data/drift_comparison_table.csv", index=False)
print(f"\n✓ Table saved to {output_dir}/data/drift_comparison_table.csv")

## 12. Visualization: Scatter Plots (Projected vs Full-Retrain)

In [ ]:
# Plot first 6 dimensions as examples
n_dims_to_plot = 6

fig, axes = plt.subplots(2, n_dims_to_plot, figsize=(20, 8))

# AE scatter plots (top row)
for dim in range(n_dims_to_plot):
    ax = axes[0, dim]
    
    # Scatter: projected vs full-retrain
    ax.scatter(
        ae_drift['aligned_projected'][:, dim],
        ae_holdout_full[:, dim],
        alpha=0.3,
        s=5,
        color='steelblue'
    )
    
    # Diagonal line (perfect agreement)
    lims = [
        min(ae_drift['aligned_projected'][:, dim].min(), ae_holdout_full[:, dim].min()),
        max(ae_drift['aligned_projected'][:, dim].max(), ae_holdout_full[:, dim].max())
    ]
    ax.plot(lims, lims, 'r--', alpha=0.5, linewidth=2)
    
    corr = ae_drift['dim_correlations'][dim]
    ax.set_title(f'AE Dim {dim+1}\nr={corr:.4f}', fontsize=10, fontweight='bold')
    ax.set_xlabel('Projected (90%)', fontsize=9)
    ax.set_ylabel('Full-Retrain (100%)', fontsize=9)
    ax.grid(True, alpha=0.3)

# UMAP scatter plots (bottom row)
for dim in range(n_dims_to_plot):
    ax = axes[1, dim]
    
    # Scatter: projected vs full-retrain
    ax.scatter(
        umap_drift['aligned_projected'][:, dim],
        umap_holdout_full[:, dim],
        alpha=0.3,
        s=5,
        color='coral'
    )
    
    # Diagonal line (perfect agreement)
    lims = [
        min(umap_drift['aligned_projected'][:, dim].min(), umap_holdout_full[:, dim].min()),
        max(umap_drift['aligned_projected'][:, dim].max(), umap_holdout_full[:, dim].max())
    ]
    ax.plot(lims, lims, 'r--', alpha=0.5, linewidth=2)
    
    corr = umap_drift['dim_correlations'][dim]
    ax.set_title(f'UMAP Dim {dim+1}\nr={corr:.4f}', fontsize=10, fontweight='bold')
    ax.set_xlabel('Projected (90%)', fontsize=9)
    ax.set_ylabel('Full-Retrain (100%)', fontsize=9)
    ax.grid(True, alpha=0.3)

plt.suptitle('Projection Drift: Projected (90%) vs Full-Retrain (100%)\nFirst 6 Dimensions', 
             fontsize=16, fontweight='bold', y=0.995)
plt.tight_layout()
plt.savefig(f"{output_dir}/projection_drift_scatter.png", dpi=300, bbox_inches='tight')
plt.show()

print(f"✓ Scatter plots saved to {output_dir}/projection_drift_scatter.png")

## 13. Visualization: Distance Distributions

In [ ]:
# Plot distribution of drift distances
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
axes[0].hist(ae_drift['distances_per_oa'], bins=50, alpha=0.6, color='steelblue', 
             label=f'AE (mean={ae_drift["mean_euclidean_distance"]:.4f})', edgecolor='black')
axes[0].hist(umap_drift['distances_per_oa'], bins=50, alpha=0.6, color='coral', 
             label=f'UMAP (mean={umap_drift["mean_euclidean_distance"]:.4f})', edgecolor='black')
axes[0].axvline(ae_drift['mean_euclidean_distance'], color='blue', linestyle='--', linewidth=2)
axes[0].axvline(umap_drift['mean_euclidean_distance'], color='red', linestyle='--', linewidth=2)
axes[0].set_xlabel('Euclidean Distance (Projected vs Full-Retrain)', fontsize=12)
axes[0].set_ylabel('Frequency', fontsize=12)
axes[0].set_title('Distribution of Drift Distances', fontsize=14, fontweight='bold')
axes[0].legend(fontsize=11)
axes[0].grid(True, alpha=0.3)

# Box plot
data_to_plot = [ae_drift['distances_per_oa'], umap_drift['distances_per_oa']]
bp = axes[1].boxplot(data_to_plot, labels=['AE', 'UMAP'], patch_artist=True)
bp['boxes'][0].set_facecolor('steelblue')
bp['boxes'][1].set_facecolor('coral')
axes[1].set_ylabel('Euclidean Distance', fontsize=12)
axes[1].set_title('Drift Distance Distribution (Box Plot)', fontsize=14, fontweight='bold')
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(f"{output_dir}/drift_distance_distributions.png", dpi=300, bbox_inches='tight')
plt.show()

print(f"✓ Distribution plots saved to {output_dir}/drift_distance_distributions.png")

## 14. Visualization: Per-Dimension Correlation Comparison

In [ ]:
# Plot per-dimension correlations
fig, ax = plt.subplots(figsize=(14, 6))

dims = np.arange(latent_dim)
width = 0.35

ax.bar(dims - width/2, ae_drift['dim_correlations'], width, 
       label='AE', color='steelblue', alpha=0.7, edgecolor='black')
ax.bar(dims + width/2, umap_drift['dim_correlations'], width,
       label='UMAP', color='coral', alpha=0.7, edgecolor='black')

ax.axhline(y=1.0, color='green', linestyle='--', linewidth=2, alpha=0.5, label='Perfect correlation')
ax.axhline(y=ae_drift['mean_correlation'], color='blue', linestyle='--', linewidth=1.5, alpha=0.7)
ax.axhline(y=umap_drift['mean_correlation'], color='red', linestyle='--', linewidth=1.5, alpha=0.7)

ax.set_xlabel('Dimension', fontsize=12)
ax.set_ylabel('Correlation (Projected vs Full-Retrain)', fontsize=12)
ax.set_title(f'Per-Dimension Correlation: Projected vs Full-Retrain Embeddings\n'
             f'AE mean={ae_drift["mean_correlation"]:.4f}, UMAP mean={umap_drift["mean_correlation"]:.4f}', 
             fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3, axis='y')
ax.set_ylim([0, 1.05])

plt.tight_layout()
plt.savefig(f"{output_dir}/per_dimension_correlations.png", dpi=300, bbox_inches='tight')
plt.show()

print(f"✓ Per-dimension correlation plot saved to {output_dir}/per_dimension_correlations.png")

## 15. Summary and Interpretation

In [ ]:
print("=" * 80)
print("EXPERIMENT SUMMARY: OUT-OF-SAMPLE PROJECTION DRIFT")
print("=" * 80)

print(f"\n📊 Experimental Setup:")
print(f"  - Held out: {len(X_holdout_10)} OAs ({holdout_fraction*100:.0f}%)")
print(f"  - Training: {len(X_train_90)} OAs ({(1-holdout_fraction)*100:.0f}%)")
print(f"  - Embedding dimension: {latent_dim}D")

print(f"\n📈 Key Findings:")

distance_ratio = umap_drift['mean_euclidean_distance'] / ae_drift['mean_euclidean_distance']
print(f"\n1. Mean Drift Distance:")
print(f"   AE:   {ae_drift['mean_euclidean_distance']:.6f}")
print(f"   UMAP: {umap_drift['mean_euclidean_distance']:.6f}")
print(f"   → UMAP drifts {distance_ratio:.1f}x more than AE")

print(f"\n2. Mean Correlation (Projected vs Full-Retrain):")
print(f"   AE:   {ae_drift['mean_correlation']:.6f}")
print(f"   UMAP: {umap_drift['mean_correlation']:.6f}")
print(f"   → AE is {ae_drift['mean_correlation'] - umap_drift['mean_correlation']:.4f} more correlated")

print(f"\n3. Procrustes Disparity:")
print(f"   AE:   {ae_drift['procrustes_disparity']:.6f}")
print(f"   UMAP: {umap_drift['procrustes_disparity']:.6f}")

print(f"\n💡 Interpretation:")
if ae_drift['mean_correlation'] > 0.95:
    print(f"   ✓ AE shows EXCELLENT stability (r > 0.95)")
    print(f"     Projected embeddings are nearly identical to full-retrain")
elif ae_drift['mean_correlation'] > 0.90:
    print(f"   ✓ AE shows GOOD stability (r > 0.90)")
else:
    print(f"   ⚠ AE shows MODERATE stability (r = {ae_drift['mean_correlation']:.3f})")

if umap_drift['mean_correlation'] < 0.80:
    print(f"   ✗ UMAP shows POOR stability (r < 0.80)")
    print(f"     Global manifold shifts significantly when adding new data")
elif umap_drift['mean_correlation'] < 0.90:
    print(f"   ⚠ UMAP shows MODERATE stability (r = {umap_drift['mean_correlation']:.3f})")
else:
    print(f"   ✓ UMAP shows GOOD stability (r > 0.90)")

print(f"\n🎯 Conclusion:")
print(f"   AE handles out-of-sample projection {distance_ratio:.1f}x better than UMAP.")
print(f"   This demonstrates AE's suitability for scenarios where new data")
print(f"   (e.g., Scotland census) needs to be added to existing national indicators.")

print(f"\n📁 All results saved to: {output_dir}")
print("=" * 80)